# Data Understanding

Explore Chicago Crime dataset schema, volume, and column distributions.

CSV/Delta verisinin kolonları, örnek satırları, veri hacmi ve temel alanları anlaşılır.

# 01 - Data Understanding

Bu notebook, Chicago Crime veri setinin temel yapısını, kolonlarını ve proje kapsamında kullanılacak hedef değişkenleri açıklar.

Bu projede amaç:

- Suç tipi tahmini
- Bölge / district tahmini

için büyük veri pipeline'ı oluşturmaktır.

## Veri seti

Kullanılan veri seti Chicago Crime Data veri setidir.

Ana alanlar:

- `id`: kayıt ID
- `case_number`: olay/case numarası
- `date`: olay tarihi
- `primary_type`: suç tipi
- `description`: suç açıklaması
- `location_description`: olay yeri tipi
- `arrest`: tutuklama bilgisi
- `domestic`: domestic olay bilgisi
- `district`: polis bölgesi
- `ward`: belediye bölgesi
- `community_area`: community area bilgisi
- `latitude`, `longitude`: konum bilgileri

In [1]:
import pandas as pd

csv_path = "../data/raw/chicago_crimes_sample.csv"

df = pd.read_csv(csv_path)
df.head()

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,year,updated_on,x_coordinate,y_coordinate,latitude,longitude,location
0,14182688,JK236765,2026-04-24T00:00:00.000,014XX S KOMENSKY AVE,1562,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,RESIDENCE,False,False,...,24,29.0,17,2026,2026-05-01T15:43:47.000,NaN,NaN,NaN,NaN,NaN
1,14179833,JK233487,2026-04-24T00:00:00.000,038XX S STATE ST,0890,THEFT,FROM BUILDING,APARTMENT,False,False,...,3,35.0,06,2026,2026-05-01T15:43:47.000,1176927.0,1879494.0,41.824674,-87.626413,"{'latitude': '41.82467357', 'longitude': '-87...."
2,14176363,JK229163,2026-04-24T00:00:00.000,038XX W ADAMS ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,False,True,...,28,26.0,08B,2026,2026-05-01T15:43:47.000,1150796.0,1898737.0,41.878029,-87.721778,"{'latitude': '41.87802873', 'longitude': '-87...."
3,14182062,JK236215,2026-04-24T00:00:00.000,018XX N WINNEBAGO AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,False,False,...,32,22.0,06,2026,2026-05-01T15:43:47.000,1160638.0,1912071.0,41.914420,-87.685270,"{'latitude': '41.914420198', 'longitude': '-87..."
4,14179693,JK233132,2026-04-24T00:00:00.000,112XX S WALLACE ST,0430,BATTERY,AGGRAVATED - OTHER DANGEROUS WEAPON,SCHOOL - PUBLIC GROUNDS,False,False,...,21,49.0,04B,2026,2026-05-01T15:43:47.000,1174313.0,1830380.0,41.689957,-87.637461,"{'latitude': '41.68995741', 'longitude': '-87...."


In [2]:
df.shape

(100000, 22)

In [3]:
df.columns.tolist()

['id',
 'case_number',
 'date',
 'block',
 'iucr',
 'primary_type',
 'description',
 'location_description',
 'arrest',
 'domestic',
 'beat',
 'district',
 'ward',
 'community_area',
 'fbi_code',
 'year',
 'updated_on',
 'x_coordinate',
 'y_coordinate',
 'latitude',
 'longitude',
 'location']

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    100000 non-null  int64  
 1   case_number           100000 non-null  str    
 2   date                  100000 non-null  str    
 3   block                 100000 non-null  str    
 4   iucr                  100000 non-null  str    
 5   primary_type          100000 non-null  str    
 6   description           100000 non-null  str    
 7   location_description  99502 non-null   str    
 8   arrest                100000 non-null  bool   
 9   domestic              100000 non-null  bool   
 10  beat                  100000 non-null  int64  
 11  district              100000 non-null  int64  
 12  ward                  100000 non-null  int64  
 13  community_area        99997 non-null   float64
 14  fbi_code              100000 non-null  str    
 15  year        

In [5]:
df[["id", "case_number", "date", "primary_type", "description", "location_description", "district", "arrest", "domestic"]].head()

,id,case_number,date,primary_type,description,location_description,district,arrest,domestic
0,14182688,JK236765,2026-04-24T00:00:00.000,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,RESIDENCE,10,False,False
1,14179833,JK233487,2026-04-24T00:00:00.000,THEFT,FROM BUILDING,APARTMENT,2,False,False
2,14176363,JK229163,2026-04-24T00:00:00.000,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,11,False,True
3,14182062,JK236215,2026-04-24T00:00:00.000,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,14,False,False
4,14179693,JK233132,2026-04-24T00:00:00.000,BATTERY,AGGRAVATED - OTHER DANGEROUS WEAPON,SCHOOL - PUBLIC GROUNDS,22,False,False


## Hedef değişkenler

Bu projede iki temel tahmin problemi üzerinde durulabilir:

1. **Suç tipi tahmini**
   - Hedef değişken: `primary_type`

2. **Bölge tahmini**
   - Hedef değişken: `district`

İlk aşamada sınıflandırma problemi olarak `primary_type` tahmini yapılacaktır. Bölge tahmini ise ikinci hedef olarak değerlendirilebilir.

In [6]:
df["primary_type"].value_counts().head(10)

primary_type
THEFT                  22376
BATTERY                18177
CRIMINAL DAMAGE        11136
ASSAULT                 8842
MOTOR VEHICLE THEFT     7937
OTHER OFFENSE           7035
DECEPTIVE PRACTICE      5875
BURGLARY                5287
NARCOTICS               3003
CRIMINAL TRESPASS       2476
Name: count, dtype: int64

In [7]:
df["district"].value_counts().head(10)

district
12    6470
8     6277
2     5681
6     5628
3     5621
1     5584
18    5496
19    5475
4     5427
11    5151
Name: count, dtype: int64

In [8]:
df.isnull().sum().sort_values(ascending=False).head(15)

location_description    498
location                 66
longitude                66
latitude                 66
y_coordinate             66
x_coordinate             66
community_area            3
ward                      0
updated_on                0
year                      0
fbi_code                  0
id                        0
case_number               0
beat                      0
domestic                  0
dtype: int64

## Sonuç

Veri setinde suç tipi, olay zamanı, konum bilgisi, tutuklama durumu ve bölge bilgisi gibi alanlar bulunmaktadır.

Bu alanlar sonraki aşamalarda:

- EDA
- Feature Engineering
- Makine öğrenmesi
- Dashboard

için kullanılacaktır.